# Data existence check post-processing

The following notebook determines if data is present at the specified reaches. The user will also need to specify the downloaded SoS granules from the Confluence workflow processing that is being analyzed. 

In [ ]:
import datetime
import json
import pathlib

import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
import pandas as pd

In [ ]:
# Constants that require updating based on execution
REACHES_OF_INTEREST_FILE = pathlib.Path.cwd().joinpath("data").joinpath("reaches_sit.json")
PRIOR_SOS_GRANULE = pathlib.Path.cwd().joinpath("data").joinpath("na_sword_v16_SOS_priors.nc")
RESULT_SOS_GRANULE = pathlib.Path.cwd().joinpath("data").joinpath("na_sword_v16_SOS_results.nc")

In [ ]:
# Constants to capture algorithms and variables
FLPE = {
    "metroman": "allq",
    "momma": "Q",
    "neobam": "q",
    "sad": "Qa",
    "sic4dvar": "Q_da"
}
MOI = {
    "metroman": "q",
    "momma": "q",
    "neobam": "q",
    "sad": "q",
    "sic4dvar": "q"
}
FILL = {
    "f8": -999999999999.0,
    "i4": -999,
    "i8": -999999999999,
    "S1": "x",
    "S48": b"\x00" * 48        # 48-byte padded string fill (e.g., tile_name)
}

In [ ]:
# Retrieve reach identifiers used in processing
with open(REACHES_OF_INTEREST_FILE) as jf:
    reach_identifiers = json.load(jf)
reach_identifiers = [ int(reach_id) for reach_id in reach_identifiers ]
reach_identifiers.sort()

In [ ]:
# Open SOS files to work with underlying data
priors = nc.Dataset(PRIOR_SOS_GRANULE, format="NETCDF4")
results = nc.Dataset(RESULT_SOS_GRANULE, format="NETCDF4")

In [ ]:
# Function to retrieve SWOT observation time
def retrieve_time(reach_id, reach_index, results, track_results):
    """Retrieve SWOT observation time."""
    
    # Locate reach index
    time = results['reaches']['time'][reach_index][0]

    # Convert to a string representation of time
    swot_ts = datetime.datetime(2000,1,1,0,0,0)   # SWOT timestamp delta
    if len(time) == 1 and time[0] == FILL["f8"]:
        track_results[reach_id] = { "time_steps": np.array(["no_data"]) }
    
    # Results data was found the time data needs to be formatted
    else:
        # Assemble discharge time strings and include missing data
        discharge_time = []
        for st in time:
            if st == FILL["f8"]:
                discharge_time.append("no_data")
            else:
                discharge_time.append((swot_ts + datetime.timedelta(seconds=st)).strftime("%Y-%m-%dT%H:%M:%S"))
        track_results[reach_id] = { "time_steps": np.array(discharge_time) }

In [ ]:
# Function to retrieve discharge
def retrieve_discharge(reach_id, reach_index, results, track_results):
    """Retrieve discharge from SOS results."""

    found_data = []
    
    # Reach-level algorithms
    for algorithm, variable in FLPE.items():
        if algorithm == "neobam":
            track_results[reach_id][algorithm] = results[algorithm][variable][variable][reach_index][0]
        else:
            track_results[reach_id][algorithm] = results[algorithm][variable][reach_index][0]

        # Set missing values to NaN
        missing = np.where(track_results[reach_id][algorithm] == FILL["f8"])
        track_results[reach_id][algorithm][missing] = np.nan
        if not pd.isna(track_results[reach_id][algorithm]).all():
            found_data.append(True)
        else:
            found_data.append(False)

    # Basin-level algorithms
    track_results[reach_id]["moi"] = {}
    for algorithm, variable in MOI.items():
        track_results[reach_id]["moi"][algorithm] = results["moi"][algorithm][variable][reach_index][0]
        missing = np.where(track_results[reach_id]["moi"][algorithm] == FILL["f8"])
        track_results[reach_id]["moi"][algorithm][missing] = np.nan
        if not pd.isna(track_results[reach_id][algorithm]).all():
            found_data.append(True)
        else:
            found_data.append(False)

    return np.array(found_data)

In [ ]:
# Retrieve SWOT observation times
track_results = {}
plot_results = {}
results_with_data = 0
for reach_id in reach_identifiers:
    
    # Locate index of reach
    reach_index = np.where(results['reaches']['reach_id'][:] == reach_id)

    # Retrieve SWOT observation times
    retrieve_time(reach_id, reach_index, results, track_results)

    # Locate algorithm results for data indexes
    found_data = retrieve_discharge(reach_id, reach_index, results, track_results)
    if found_data.any():
        results_with_data += 1
    if np.count_nonzero(found_data) >= found_data.size / 2:
        plot_results[reach_id] = track_results[reach_id]

print("Number of reach identifiers found: ", len(track_results.keys()))
print("Number of results with data: ", results_with_data)
print("Number of results where more than half the algorithms produced data: ", len(plot_results.keys()))

In [ ]:
# Select reaches to plot
# plot_keys = list(plot_results.keys())
# key_1 = plot_keys[0]
# key_2 = plot_keys[len(plot_keys)//2]
# key_3 = plot_keys[-1]

REACH_1 = 74261000011
REACH_2 = 74267300011
REACH_3 = 74269900751
REACHES = [REACH_1, REACH_2, REACH_3]

In [ ]:
# Function to plot discharge
def plot_discharge(axs, discharge_time, discharge_q, color, name, line_list, line_list_names):
    """Plot discharge values and return list of plot lines."""

    if np.all(np.isnan(discharge_q)):
        return   # Cannot plot NaN values
    
    else:
        axs.scatter(discharge_time, discharge_q, color=color)
        if "MOI" in name:
            line, = axs.plot(discharge_time, discharge_q, color=color, linestyle="dashed")
        else:
            line, = axs.plot(discharge_time, discharge_q, color=color)
        line_list.append(line)
        line_list_names.append(name)

In [ ]:
# Plot REACH #1
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize = (25,15))

for idx, ax in enumerate((ax1, ax2, ax3)):

    reach_id = REACHES[idx]

    ## Build legend data
    line_list = []
    line_list_names = []
    
    # FLPE & MOI
    time = plot_results[reach_id]["time_steps"]
    plot_discharge(ax, time, plot_results[reach_id]["metroman"], "orange", "MetroMan", line_list, line_list_names)
    plot_discharge(ax, time, plot_results[reach_id]["moi"]["metroman"], "orange", "MOI MetroMan", line_list, line_list_names)
    plot_discharge(ax, time, plot_results[reach_id]["momma"], "red", "MOMMA", line_list, line_list_names)
    plot_discharge(ax, time, plot_results[reach_id]["moi"]["momma"], "red", "MOI MOMMA", line_list, line_list_names)
    plot_discharge(ax, time, plot_results[reach_id]["neobam"], "purple", "NeoBAM", line_list, line_list_names)
    plot_discharge(ax, time, plot_results[reach_id]["moi"]["neobam"], "purple", "MOI NeoBAM", line_list, line_list_names)
    plot_discharge(ax, time, plot_results[reach_id]["sad"], "green", "SAD", line_list, line_list_names)
    plot_discharge(ax, time, plot_results[reach_id]["moi"]["sad"], "green", "MOI SAD", line_list, line_list_names)
    plot_discharge(ax, time, plot_results[reach_id]["sic4dvar"], "blue", "SIC4DVar", line_list, line_list_names)
    plot_discharge(ax, time, plot_results[reach_id]["moi"]["sic4dvar"], "blue", "MOI SIC4DVar", line_list, line_list_names)
    
    # Plot details
    ax.legend(line_list, line_list_names, loc="center right")
    ax.set_ylabel("Discharge", fontsize=10)
    ax.set_xlabel("Time", fontsize=8)
    ax.set_xticks(time) # Set tick locations
    ax.set_xticklabels(time, rotation=90) # Set labels and rotate
    ax.set_title(f"{reach_id} Discharge", fontsize=10)

# Save file
save_file = f"data_check_algos_{datetime.datetime.now().strftime("%Y%m%d")}.png"
save_file_path = pathlib.Path.cwd().joinpath("data").joinpath("plots").joinpath(save_file)
save_file_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(save_file_path, dpi=300)

# Display
plt.tight_layout()
plt.show()

# Locate gauge data

The next section locates gauge data so that it might be plotted next to discharge for comparison.

In [ ]:
# Locate reaches that have gauges from the above reaches of interest
GAUGE_AGENCY = "USGS"
gauge_ids = priors[GAUGE_AGENCY][f"{GAUGE_AGENCY}_reach_id"][:]
print(f"Number of Gauge reach identifiers: {len(gauge_ids)}")

reach_overlap = np.intersect1d(gauge_ids, np.array(list(plot_results.keys())))
print(f"Number of overlapping reaches: {len(reach_overlap)}")

In [ ]:
# Locate data for these reaches
gauge_data = {}
for reach_id in reach_overlap:

    # Get gauge discharge and filter out missing data
    reach_gauge_index = np.where(gauge_ids == reach_id)
    gauge_discharge = priors[GAUGE_AGENCY][f"{GAUGE_AGENCY}_q"][reach_gauge_index].filled()[0]

    missing = np.where(gauge_discharge == FILL["f8"])
    gauge_discharge[missing] = np.nan

    # Get time and filter out missing values
    time = priors[GAUGE_AGENCY][f"{GAUGE_AGENCY}_qt"][reach_gauge_index].filled().astype(int)[0]
    gauge_time = np.array([ datetime.datetime.fromordinal(gt).strftime("%Y-%m-%d") for gt in time if gt != -999999999999 ])

    # Match discharge time to gauge time
    time = plot_results[reach_id]["time_steps"]
    updated_discharge_time = np.array([ st.split("T")[0] for st in time ])

    time_intersection, gauge_idx, discharge_idx = np.intersect1d(gauge_time, updated_discharge_time, return_indices=True)
    gauge_discharge = gauge_discharge[gauge_idx]
    # Only save gauge if data is present
    if not np.all(np.isnan(gauge_discharge)):
        gauge_data[reach_id] = {
            "time_steps": gauge_time[gauge_idx],
            "discharge_idx": discharge_idx,
            "gauge_discharge": gauge_discharge
        }

print(f"Total number of reach identifiers with gauge data: {len(gauge_data.keys())}")

In [ ]:
# Plot gauge data for three reaches
# gauge_keys = list(gauge_data.keys())
# key_1 = gauge_keys[0]
# key_2 = gauge_keys[len(gauge_keys)//2]
# key_3 = gauge_keys[-1]
# print(key_1, key_2, key_3)

REACH_1 = 74262200031
REACH_2 = 74267600061
REACH_3 = 74269900481
REACHES = [REACH_1, REACH_2, REACH_3]

In [ ]:
# Function to plot discharge
def plot_discharge(axs, discharge_time, discharge_q, color, name, line_list, line_list_names):
    """Plot discharge values and return list of plot lines."""

    if np.all(np.isnan(discharge_q)):
        return   # Cannot plot NaN values
    
    else:
        axs.scatter(discharge_time, discharge_q, color=color)
        if "MOI" in name:
            line, = axs.plot(discharge_time, discharge_q, color=color, linestyle="dashed")
        else:
            line, = axs.plot(discharge_time, discharge_q, color=color)
        line_list.append(line)
        line_list_names.append(name)

In [ ]:
# Plot REACH #1
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize = (25,15))

for idx, ax in enumerate((ax1, ax2, ax3)):

    reach_id = REACHES[idx]

    # Get indexes to match discharge and gauge time steps
    gauge_time = gauge_data[reach_id]["time_steps"]
    discharge_idx = gauge_data[reach_id]["discharge_idx"]
    # Isolate discharge time to day
    discharge_time = np.array([ dt.split("T")[0] for dt in plot_results[reach_id]["time_steps"][discharge_idx] ])

    # Build legend data
    line_list = []
    line_list_names = []

    # Gauge
    plot_discharge(ax, gauge_time, gauge_data[reach_id]["gauge_discharge"], "grey", "Gauge", line_list, line_list_names)
    
    # FLPE & MOI
    plot_discharge(ax, discharge_time, plot_results[reach_id]["metroman"][discharge_idx], "orange", "MetroMan", line_list, line_list_names)
    plot_discharge(ax, discharge_time, plot_results[reach_id]["moi"]["metroman"][discharge_idx], "orange", "MOI MetroMan", line_list, line_list_names)
    plot_discharge(ax, discharge_time, plot_results[reach_id]["momma"][discharge_idx], "red", "MOMMA", line_list, line_list_names)
    plot_discharge(ax, discharge_time, plot_results[reach_id]["moi"]["momma"][discharge_idx], "red", "MOI MOMMA", line_list, line_list_names)
    plot_discharge(ax, discharge_time, plot_results[reach_id]["neobam"][discharge_idx], "purple", "NeoBAM", line_list, line_list_names)
    plot_discharge(ax, discharge_time, plot_results[reach_id]["moi"]["neobam"][discharge_idx], "purple", "MOI NeoBAM", line_list, line_list_names)
    plot_discharge(ax, discharge_time, plot_results[reach_id]["sad"][discharge_idx], "green", "SAD", line_list, line_list_names)
    plot_discharge(ax, discharge_time, plot_results[reach_id]["moi"]["sad"][discharge_idx], "green", "MOI SAD", line_list, line_list_names)
    plot_discharge(ax, discharge_time, plot_results[reach_id]["sic4dvar"][discharge_idx], "blue", "SIC4DVar", line_list, line_list_names)
    plot_discharge(ax, discharge_time, plot_results[reach_id]["moi"]["sic4dvar"][discharge_idx], "blue", "MOI SIC4DVar", line_list, line_list_names)
    
    # Plot details
    ax.legend(line_list, line_list_names, loc="center right")
    ax.set_ylabel("Discharge", fontsize=10)
    ax.set_xlabel("Time", fontsize=8)
    ax.set_xticks(discharge_time) # Set tick locations
    ax.set_xticklabels(discharge_time, rotation=90) # Set labels and rotate
    ax.set_title(f"{reach_id} Discharge", fontsize=10)

# Save file
save_file = f"data_check_gauge_{datetime.datetime.now().strftime("%Y%m%d")}.png"
save_file_path = pathlib.Path.cwd().joinpath("data").joinpath("plots").joinpath(save_file)
plt.savefig(save_file_path, dpi=300, bbox_inches="tight")

# Display
plt.tight_layout()
plt.show()

In [ ]:
# Close SOS files
priors.close()
results.close()